# PySpark DataFrame Transformations and Actions 



It includes:

- clear definitions
- lazy vs eager explanation
- good sample DataFrames
- major DataFrame transformations
- major DataFrame actions
- joins, aggregations, null handling, window functions
- nested data and complex columns
- partitioning and performance-related operations
- SQL view usage
- write actions
- notes on what is a transformation vs what is a column function

## Important note

Spark has **hundreds of helper functions** in `pyspark.sql.functions` (string, date, math, JSON, array, map, struct, regex, hashing, etc.).  
This notebook covers the **major DataFrame transformations and actions you are expected to know and use in real projects, interviews.


## 1. What is a DataFrame?

A Spark DataFrame is a distributed table with:
- rows
- columns
- schema
- partitioned execution in a cluster

You can think of it like:
- a SQL table
- a Pandas DataFrame
- but distributed across many machines

## 2. What is a Transformation?

A transformation returns a **new DataFrame** instead of executing immediately.

Examples:
- `select()`
- `filter()`
- `withColumn()`
- `join()`
- `groupBy().agg()`
- `dropDuplicates()`
- `repartition()`

These are **lazy**. Spark only builds an execution plan first.

## 3. What is an Action?

An action triggers actual execution and returns:
- result to driver
- output to screen
- output to storage
- some summary

Examples:
- `show()`
- `count()`
- `collect()`
- `first()`
- `take()`
- `write.parquet(...)`

## 4. Very important distinction

Some things are not standalone DataFrame transformations but are commonly used inside transformations:

- `col()`
- `lit()`
- `when()`
- `expr()`
- `cast()`
- `alias()`
- `asc()`, `desc()`
- `row_number()`, `rank()`, `lag()`, `lead()`

These are usually **column expressions / helper functions**, and they are applied inside transformations such as `select`, `withColumn`, `agg`, `orderBy`, or `over`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Complete_DataFrame_Transformations_Actions")
    .master("local[*]")   
    .getOrCreate()
)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 21:31:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 5. Create strong sample DataFrames

We will create:
- employee DataFrame
- department DataFrame
- bonus DataFrame
- nested/array DataFrame
- duplicate/null DataFrame
- union/set-operation DataFrames

This allows us to demonstrate nearly all important operations in one place.

In [2]:
employee_data = [
    (101, "Anuj",   "IT",      90000, 29, "Mumbai",    1001, "2023-01-10", 4.7, None),
    (102, "Riya",   "HR",      60000, 31, "Pune",      1002, "2022-11-05", 4.2, "A"),
    (103, "Vikas",  "IT",      85000, 27, "Bengaluru", 1001, "2024-02-12", 4.5, "B"),
    (104, "Sneha",  "Finance", 95000, 35, "Mumbai",    1003, "2021-08-19", 4.8, "A"),
    (105, "Amit",   "Sales",   50000, 26, "Delhi",     1004, "2023-06-01", 3.9, None),
    (106, "Pooja",  "HR",      65000, 30, "Chennai",   1002, "2020-12-11", 4.1, "B"),
    (107, "Karan",  "IT",     120000, 33, "Hyderabad", 1001, "2019-03-14", 4.9, "A"),
    (108, "Meera",  "Sales",   52000, 28, "Pune",      1004, "2024-01-18", 3.7, "C"),
    (109, "Rohit",  "Finance", 88000, 32, "Delhi",     1003, "2022-09-09", 4.0, "B"),
    (110, "Nisha",  "Ops",     70000, 29, "Mumbai",    1005, "2023-04-22", 4.3, None),
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), False),
    StructField("dept", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("manager_id", IntegerType(), True),
    StructField("join_date", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("grade", StringType(), True),
])

emp_df = spark.createDataFrame(employee_data, employee_schema) \
    .withColumn("join_date", F.to_date("join_date"))

dept_data = [
    ("IT", "Technology", "North"),
    ("HR", "Human Resources", "West"),
    ("Finance", "Finance & Accounts", "North"),
    ("Sales", "Business Sales", "South"),
    ("Ops", "Operations", "West"),
    ("Legal", "Legal Affairs", "East"),
]

dept_df = spark.createDataFrame(dept_data, ["dept", "dept_full_name", "region"])

bonus_data = [
    (101, 15000, "2024"),
    (103, 12000, "2024"),
    (104, 20000, "2024"),
    (107, 30000, "2024"),
    (111, 10000, "2024"),
]
bonus_df = spark.createDataFrame(bonus_data, ["emp_id", "bonus_amount", "bonus_year"])

nested_data = [
    (1, ["python", "spark", "aws"], {"laptop": "dell", "level": "senior"}, ("A", 10)),
    (2, ["sql", "powerbi"], {"laptop": "hp", "level": "mid"}, ("B", 20)),
    (3, [], {"laptop": "mac", "level": "lead"}, ("C", 30)),
]
nested_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("skills", ArrayType(StringType()), True),
    StructField("profile_map", MapType(StringType(), StringType()), True),
    StructField("code_struct", StructType([
        StructField("code", StringType(), True),
        StructField("score", IntegerType(), True),
    ]), True),
])
nested_df = spark.createDataFrame(nested_data, nested_schema)

null_dup_data = [
    (1, "A", 10),
    (1, "A", 10),
    (2, None, 20),
    (3, "C", None),
    (4, None, None),
]
null_dup_df = spark.createDataFrame(null_dup_data, ["id", "category", "value"])

left_df = spark.createDataFrame([(1, "x"), (2, "y"), (2, "y"), (3, "z")], ["id", "tag"])
right_df = spark.createDataFrame([(2, "y"), (3, "z"), (4, "w")], ["id", "tag"])

In [4]:
nested_df.show(truncate=False)

+---+--------------------+---------------------------------+-----------+
|id |skills              |profile_map                      |code_struct|
+---+--------------------+---------------------------------+-----------+
|1  |[python, spark, aws]|{laptop -> dell, level -> senior}|{A, 10}    |
|2  |[sql, powerbi]      |{laptop -> hp, level -> mid}     |{B, 20}    |
|3  |[]                  |{laptop -> mac, level -> lead}   |{C, 30}    |
+---+--------------------+---------------------------------+-----------+



In [3]:
print("Employees")
emp_df.show(truncate=False)

print("Departments")
dept_df.show(truncate=False)

print("Bonuses")
bonus_df.show(truncate=False)

print("Nested")
nested_df.show(truncate=False)

print("Null/Duplicates")
null_dup_df.show(truncate=False)

Employees


+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|dept   |salary|age|city     |manager_id|join_date |rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|101   |Anuj    |IT     |90000 |29 |Mumbai   |1001      |2023-01-10|4.7   |NULL |
|102   |Riya    |HR     |60000 |31 |Pune     |1002      |2022-11-05|4.2   |A    |
|103   |Vikas   |IT     |85000 |27 |Bengaluru|1001      |2024-02-12|4.5   |B    |
|104   |Sneha   |Finance|95000 |35 |Mumbai   |1003      |2021-08-19|4.8   |A    |
|105   |Amit    |Sales  |50000 |26 |Delhi    |1004      |2023-06-01|3.9   |NULL |
|106   |Pooja   |HR     |65000 |30 |Chennai  |1002      |2020-12-11|4.1   |B    |
|107   |Karan   |IT     |120000|33 |Hyderabad|1001      |2019-03-14|4.9   |A    |
|108   |Meera   |Sales  |52000 |28 |Pune     |1004      |2024-01-18|3.7   |C    |
|109   |Rohit   |Finance|88000 |32 |Delhi    |1003      |2022-09-09|4.0   |B    |
|110   |Nisha   

# 6. Core column/helper expressions used inside transformations

Before transformations, learn these very common building blocks:
- `col`
- `lit`
- `expr`
- `when`
- `otherwise`
- `alias`
- `cast`

These are not usually treated as standalone DataFrame transformations, but you use them everywhere.

In [ ]:
emp_df.select(
    F.col("emp_id").alias("employee_id"),
    F.col("emp_name"),
    F.lit("ACTIVE").alias("status"),
    F.expr("salary * 0.10").alias("ten_percent_bonus"),
    F.when(F.col("salary") >= 90000, "High").otherwise("Standard").alias("salary_band"),
    F.col("salary").cast("double").alias("salary_double")
).show(truncate=False)

In [21]:
from pyspark.sql.functions import col,lit
# col("column name")
#emp_df.select(col("salary")>60000).show()
col("salary")



Column<'salary'>

# 7. Basic DataFrame Transformations

## 7.1 `select()`

Used to choose specific columns.

df.select("col1","col2")

df.select("name","salary")
df.select(col("name"),col("salary"))

### Why used
In real pipelines you rarely need all columns. `select()` reduces data movement and keeps logic cleaner.

In [19]:
emp_df.select("*").show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   107|   Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
|   108|   Meera|  Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   109|   Rohit|Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
|   110|   Nisha

In [16]:
emp_df.select(col("emp_name"),(col("salary")*0.1).alias("bonus")).show()

+--------+-------+
|emp_name|  bonus|
+--------+-------+
|    Anuj| 9000.0|
|    Riya| 6000.0|
|   Vikas| 8500.0|
|   Sneha| 9500.0|
|    Amit| 5000.0|
|   Pooja| 6500.0|
|   Karan|12000.0|
|   Meera| 5200.0|
|   Rohit| 8800.0|
|   Nisha| 7000.0|
+--------+-------+



In [14]:
emp_df.select("emp_id", "emp_name", "dept", "salary").show()

+------+--------+-------+------+
|emp_id|emp_name|   dept|salary|
+------+--------+-------+------+
|   101|    Anuj|     IT| 90000|
|   102|    Riya|     HR| 60000|
|   103|   Vikas|     IT| 85000|
|   104|   Sneha|Finance| 95000|
|   105|    Amit|  Sales| 50000|
|   106|   Pooja|     HR| 65000|
|   107|   Karan|     IT|120000|
|   108|   Meera|  Sales| 52000|
|   109|   Rohit|Finance| 88000|
|   110|   Nisha|    Ops| 70000|
+------+--------+-------+------+



## 7.2 `selectExpr()`

Like `select()`, but allows SQL expressions directly.

In [17]:
emp_df.selectExpr(
    "emp_id",
    "emp_name",
    "salary",
    "salary * 0.15 as allowance",
    "upper(dept) as dept_upper"
).show()

+------+--------+------+---------+----------+
|emp_id|emp_name|salary|allowance|dept_upper|
+------+--------+------+---------+----------+
|   101|    Anuj| 90000| 13500.00|        IT|
|   102|    Riya| 60000|  9000.00|        HR|
|   103|   Vikas| 85000| 12750.00|        IT|
|   104|   Sneha| 95000| 14250.00|   FINANCE|
|   105|    Amit| 50000|  7500.00|     SALES|
|   106|   Pooja| 65000|  9750.00|        HR|
|   107|   Karan|120000| 18000.00|        IT|
|   108|   Meera| 52000|  7800.00|     SALES|
|   109|   Rohit| 88000| 13200.00|   FINANCE|
|   110|   Nisha| 70000| 10500.00|       OPS|
+------+--------+------+---------+----------+



## 7.3 `withColumn()`

dg.withColumn("new_column",expression)



Add a new column or replace an existing one.

In [25]:
emp_df.withColumn("salary_after_hike", F.col("salary") * 1.10) \
      .withColumn("is_mumbai", F.when(F.col("city") == "Mumbai", True).otherwise(False)) \
      .withColumn("Test1",lit("Test lit word"))\
      .withColumn("test2",lit(""))\
      .show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+------------------+---------+-------------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade| salary_after_hike|is_mumbai|        Test1|test2|
+------+--------+-------+------+---+---------+----------+----------+------+-----+------------------+---------+-------------+-----+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL| 99000.00000000001|     true|Test lit word|     |
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|           66000.0|    false|Test lit word|     |
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B| 93500.00000000001|    false|Test lit word|     |
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|104500.00000000001|     true|Test lit word|     |
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL| 5

## 7.4 `withColumnRenamed()`

df.withColumnRenamed("old_name","new_column")

Rename a column.

In [28]:
df=emp_df.withColumnRenamed("emp_name", "employee_name")

In [29]:
mapping={
    "employee_name":"emp_new",
    "dept":"dept_new"
}

for old,new in mapping.items():
    df=df.withColumnRenamed(old,new)

In [30]:
df.show()

+------+-------+--------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_new|dept_new|salary|age|     city|manager_id| join_date|rating|grade|
+------+-------+--------+------+---+---------+----------+----------+------+-----+
|   101|   Anuj|      IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   102|   Riya|      HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   103|  Vikas|      IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   104|  Sneha| Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   105|   Amit|   Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   106|  Pooja|      HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   107|  Karan|      IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
|   108|  Meera|   Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   109|  Rohit| Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
|   110|  Nisha|

## 7.5 `drop()`

df.drop("col1","col2".....)
Remove unwanted columns.

In [ ]:
emp_df.drop("manager_id", "grade").show(5)

## 7.6 `filter()` and `where()`

df.filter(condition)

df.where(condition)

Both are the same. Used to filter rows.

In [33]:
emp_df.filter(F.col("salary") > 80000).show()

emp_df.where((F.col("dept") == "IT") & (F.col("rating") >= 4.5)).show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   107|   Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
|   109|   Rohit|Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
+------+--------+-------+------+---+---------+----------+----------+------+-----+

+------+--------+----+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+----+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|  IT| 9

## 7.7 `distinct()`

It consisder all the columns 
Returns only unique values 
Remove duplicate full rows.

In [4]:
left_df.distinct().show()

+---+---+
| id|tag|
+---+---+
|  1|  x|
|  2|  y|
|  3|  z|
+---+---+



## 7.8 `dropDuplicates()`

Remove duplicates based on all columns or selected columns.

In [ ]:
null_dup_df.dropDuplicates().show()

emp_df.select("dept", "city").dropDuplicates().show()

## 7.9 `limit()`

Restrict number of rows.

In [ ]:
emp_df.limit(3).show()

## 7.10 `sample()`

Random sampling.

In [ ]:
emp_df.sample(withReplacement=False, fraction=0.4, seed=42).show()

## 7.11 `sampleBy()`

Stratified sampling by key column.

In [ ]:
fractions = {"IT": 0.5, "HR": 1.0, "Sales": 1.0, "Finance": 0.5, "Ops": 1.0}
emp_df.sampleBy("dept", fractions=fractions, seed=42).show()

## 7.12 `sort()` / `orderBy()`

Sort records globally.

In [5]:
emp_df.sort("salary").show()

emp_df.orderBy(F.col("salary").desc(), F.col("age").asc()).show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   108|   Meera|  Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   110|   Nisha|    Ops| 70000| 29|   Mumbai|      1005|2023-04-22|   4.3| NULL|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   109|   Rohit|Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   107|   Karan

## 7.13 `sortWithinPartitions()`

Sort only inside each partition, not full global sorting.

In [ ]:
emp_df.repartition(3).sortWithinPartitions("dept", F.col()"salary".desc()).show()

In [ ]:
#emp_df.select("salary").show()
import pyspark.sql.functions as F

+------+
|salary|
+------+
| 90000|
| 60000|
| 85000|
| 95000|
| 50000|
| 65000|
|120000|
| 52000|
| 88000|
| 70000|
+------+



# 8. Aggregation Transformations

## 8.1 `groupBy()`

Creates grouped dataset.

In [ ]:
emp_df.groupBy("dept").count().show()

## 8.2 `agg()`

Apply one or more aggregate functions.

In [ ]:
emp_df.groupBy("dept").agg(
    F.count("*").alias("emp_count"),
    F.sum("salary").alias("total_salary"),
    F.avg("salary").alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary")
).orderBy("dept").show(truncate=False)

## 8.3 `rollup()`

Hierarchical aggregation. Adds subtotal and grand total rows.

In [ ]:
emp_df.rollup("dept", "city").agg(F.sum("salary").alias("total_salary")).show(truncate=False)

## 8.4 `cube()`

Multi-dimensional aggregation. Produces more subtotal combinations than rollup.

In [ ]:
emp_df.cube("dept", "city").agg(F.count("*").alias("cnt")).show(truncate=False)

## 8.5 `pivot()`

Turn row values into columns.

In [ ]:
emp_df.groupBy("city").pivot("dept").agg(F.avg("salary")).show(truncate=False)

# 9. Join Transformations

Main API: `join()`

## 9.1 Inner join

In [ ]:
emp_df.join(dept_df, on="dept", how="inner").show(truncate=False)

## 9.2 Left join

In [ ]:
emp_df.join(dept_df, on="dept", how="left").show(truncate=False)

## 9.3 Right join

In [ ]:
emp_df.join(dept_df, on="dept", how="right").show(truncate=False)

## 9.4 Full join

In [ ]:
emp_df.join(dept_df, on="dept", how="full").show(truncate=False)

## 9.5 Left semi join

Returns rows from left only where match exists in right.  
Columns from right are not included.

In [ ]:
emp_df.join(bonus_df, on="emp_id", how="left_semi").show()

## 9.6 Left anti join

Returns rows from left only where no match exists in right.

In [ ]:
emp_df.join(bonus_df, on="emp_id", how="left_anti").show()

## 9.7 Cross join

Cartesian product. Use carefully.

In [ ]:
emp_df.select("emp_id").limit(2).crossJoin(dept_df.select("dept").limit(3)).show()

## 9.8 Join using explicit condition and aliases

In [ ]:
e = emp_df.alias("e")
d = dept_df.alias("d")

e.join(d, F.col("e.dept") == F.col("d.dept"), "inner") \
 .select(
     F.col("e.emp_id"),
     F.col("e.emp_name"),
     F.col("e.dept"),
     F.col("d.dept_full_name"),
     F.col("d.region")
 ).show(truncate=False)

# 10. Set Operations

## 10.1 `union()`

Combines rows by position. Schemas must match in same order.

In [ ]:
left_df.union(right_df).show()

## 10.2 `unionByName()`

Combines rows by matching column names.

In [ ]:
a = spark.createDataFrame([(1, "x")], ["id", "tag"])
b = spark.createDataFrame([("y", 2)], ["tag", "id"])
a.unionByName(b).show()

## 10.3 `intersect()`

Common distinct rows.

In [ ]:
left_df.intersect(right_df).show()

## 10.4 `intersectAll()`

Common rows including duplicates where supported.

In [ ]:
left_df.intersectAll(right_df).show()

## 10.5 `exceptAll()`

Rows from first DataFrame that are not in second, keeping duplicate semantics.

In [ ]:
left_df.exceptAll(right_df).show()

## 10.6 `subtract()`

Another set-difference style operation often used in practice.

In [ ]:
left_df.subtract(right_df).show()

# 11. Null Handling Transformations (`DataFrameNaFunctions`)

## 11.1 `na.drop()`

In [ ]:
null_dup_df.na.drop().show()
null_dup_df.na.drop(how="all").show()
null_dup_df.na.drop(subset=["category"]).show()

## 11.2 `na.fill()`

In [ ]:
null_dup_df.na.fill({"category": "UNKNOWN", "value": 0}).show()

## 11.3 `na.replace()`

In [ ]:
emp_df.na.replace({"Mumbai": "MUM"}).show(5)

# 12. Partition / Distribution Related Transformations

These are very important in performance tuning.

## 12.1 `repartition()`

Increase partitions or reshuffle by columns.

In [ ]:
print("Original partitions:", emp_df.rdd.getNumPartitions())
rep_df = emp_df.repartition(4)
print("After repartition:", rep_df.rdd.getNumPartitions())

rep_by_dept_df = emp_df.repartition(4, "dept")
print("After repartition by dept:", rep_by_dept_df.rdd.getNumPartitions())

## 12.2 `coalesce()`

Reduce number of partitions with less shuffle than repartition.

In [ ]:
coal_df = rep_df.coalesce(2)
print("After coalesce:", coal_df.rdd.getNumPartitions())

## 12.3 `repartitionByRange()`

Range-based partitioning.

In [ ]:
range_df = emp_df.repartitionByRange(3, "salary")
print("After repartitionByRange:", range_df.rdd.getNumPartitions())

# 13. Window-based Transformations / Expressions

Window functions are applied using:
- `Window.partitionBy(...)`
- `Window.orderBy(...)`
- `.over(window_spec)`

In [ ]:
w_dept_salary = Window.partitionBy("dept").orderBy(F.col("salary").desc())
w_city_date = Window.partitionBy("city").orderBy("join_date")

## 13.1 `row_number()`

In [ ]:
emp_df.withColumn("row_num", F.row_number().over(w_dept_salary)).show()

## 13.2 `rank()`

In [ ]:
emp_df.withColumn("rank_in_dept", F.rank().over(w_dept_salary)).show()

## 13.3 `dense_rank()`

In [ ]:
emp_df.withColumn("dense_rank_in_dept", F.dense_rank().over(w_dept_salary)).show()

## 13.4 `lead()`

In [ ]:
emp_df.withColumn("next_salary", F.lead("salary").over(w_dept_salary)).show()

## 13.5 `lag()`

In [ ]:
emp_df.withColumn("previous_salary", F.lag("salary").over(w_dept_salary)).show()

## 13.6 cumulative and analytic examples

In [ ]:
w_running = Window.partitionBy("dept").orderBy("join_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

emp_df.withColumn("running_salary", F.sum("salary").over(w_running)) \
      .withColumn("avg_salary_by_dept", F.avg("salary").over(Window.partitionBy("dept"))) \
      .withColumn("first_salary_in_dept", F.first("salary").over(w_dept_salary)) \
      .withColumn("last_salary_in_dept", F.last("salary").over(w_dept_salary)) \
      .show(truncate=False)

# 14. Nested / Complex Data Transformations

## 14.1 `explode()`

Turns each element of an array into a separate row.

In [ ]:
nested_df.select("id", F.explode("skills").alias("skill")).show()

## 14.2 `explode_outer()`

Preserves rows even when array is empty or null.

In [ ]:
nested_df.select("id", F.explode_outer("skills").alias("skill")).show()

## 14.3 `posexplode()`

Like explode, but also returns position.

In [ ]:
nested_df.select("id", F.posexplode("skills").alias("position", "skill")).show()

## 14.4 `inline()`

Explodes array of structs into columns.  
We first build an example.

In [ ]:
arr_struct_df = spark.createDataFrame([
    (1, [{"k": "a", "v": 10}, {"k": "b", "v": 20}]),
    (2, [{"k": "x", "v": 99}]),
], schema="id int, items array<struct<k:string,v:int>>")

arr_struct_df.select("id", F.inline("items")).show()

## 14.5 `array()`, `struct()`, `create_map()`

In [ ]:
emp_df.select(
    "emp_id",
    F.array("dept", "city").alias("dept_city_array"),
    F.struct("emp_name", "salary").alias("emp_struct"),
    F.create_map(F.lit("dept"), F.col("dept"), F.lit("city"), F.col("city")).alias("meta_map")
).show(truncate=False)

## 14.6 Accessing struct/map/array fields

In [ ]:
nested_df.select(
    "id",
    F.col("code_struct.code").alias("code"),
    F.col("code_struct.score").alias("score"),
    F.col("profile_map")["laptop"].alias("laptop"),
    F.element_at("skills", 1).alias("first_skill")
).show(truncate=False)

# 15. Other Very Common DataFrame Transformations Used in Practice

## 15.1 `alias()` in select / joins / aggregations

In [ ]:
emp_df.groupBy("dept").agg(F.avg("salary").alias("avg_salary")).show()

## 15.2 `cache()` and `persist()`

These return a DataFrame-like object and are commonly treated as optimization-related transformations.
They do not show value until an action runs.

In [ ]:
cached_df = emp_df.filter(F.col("salary") > 60000).cache()
print("Nothing computed yet until action")
print("Count after cache action:", cached_df.count())

from pyspark import StorageLevel
persisted_df = emp_df.persist(StorageLevel.MEMORY_AND_DISK)
print("Persist count:", persisted_df.count())

## 15.3 `unpersist()`

Releases cached/persisted data.

In [ ]:
persisted_df.unpersist()
cached_df.unpersist()

## 15.4 `hint()`

Optimization hint, commonly used for broadcast joins.

In [ ]:
emp_df.hint("broadcast").join(dept_df, "dept").show()

## 15.5 `transform()` on DataFrame

Useful for reusable functional pipeline style.

In [ ]:
def add_tax_flag(df):
    return df.withColumn("tax_flag", F.when(F.col("salary") > 80000, "HIGH_TAX").otherwise("NORMAL_TAX"))

emp_df.transform(add_tax_flag).show()

## 15.6 `replace()` on DataFrame

Often used for value replacement.

In [ ]:
emp_df.replace("Pune", "PUNE_CITY", subset=["city"]).show()

# 16. SQL View Related Operations

## 16.1 `createOrReplaceTempView()`

In [ ]:
emp_df.createOrReplaceTempView("employees")
dept_df.createOrReplaceTempView("departments")

## 16.2 SQL query on DataFrames

In [ ]:
spark.sql("""
SELECT
    e.dept,
    d.region,
    COUNT(*) AS emp_count,
    AVG(e.salary) AS avg_salary
FROM employees e
LEFT JOIN departments d
    ON e.dept = d.dept
GROUP BY e.dept, d.region
ORDER BY avg_salary DESC
""").show(truncate=False)

## 16.3 `createGlobalTempView()`

Accessible using database `global_temp`.

In [ ]:
emp_df.createOrReplaceGlobalTempView("employees_global")

spark.sql("SELECT emp_id, emp_name, dept FROM global_temp.employees_global LIMIT 5").show()

# 17. Inspecting Plans and Schema

These are not actions in the same sense as `collect()`, but are very important for learning and debugging.

In [ ]:
emp_df.printSchema()

In [ ]:
emp_df.explain()

# 18. DataFrame Actions

These trigger execution.

## 18.1 `show()`

In [ ]:
emp_df.show(5, truncate=False)

## 18.2 `count()`

In [ ]:
emp_df.count()

## 18.3 `collect()`

Brings all rows to driver. Use carefully on small data only.

In [ ]:
emp_df.select("emp_id", "emp_name").collect()[:3]

## 18.4 `first()`

In [ ]:
emp_df.first()

## 18.5 `head()`

In [ ]:
emp_df.head(2)

## 18.6 `take()`

In [ ]:
emp_df.take(3)

## 18.7 `toLocalIterator()`

Returns an iterator over rows.

In [ ]:
it = emp_df.select("emp_id", "emp_name").toLocalIterator()
for i, row in enumerate(it):
    print(row)
    if i == 2:
        break

## 18.8 `foreach()`

Executes a function for each row.
Note: mainly useful for side effects.

In [ ]:
def print_emp(row):
    print(f"Employee => {row.emp_id}, {row.emp_name}, {row.dept}")

emp_df.select("emp_id", "emp_name", "dept").limit(3).foreach(print_emp)

## 18.9 `foreachPartition()`

Executes function once per partition.

In [ ]:
def process_partition(rows):
    count = 0
    for _ in rows:
        count += 1
    print("Partition row count:", count)

emp_df.repartition(2).foreachPartition(process_partition)

## 18.10 `describe()`

In [ ]:
emp_df.describe(["salary", "age", "rating"]).show()

## 18.11 `summary()`

In [ ]:
emp_df.summary().show()

## 18.12 `toPandas()`

Bring to pandas DataFrame. Use only for small datasets.

In [ ]:
pdf = emp_df.limit(5).toPandas()
pdf

## 18.13 write actions

Writing to storage triggers execution.

In [ ]:
output_base = "/mnt/data/spark_df_demo_output"

emp_df.write.mode("overwrite").json(f"{output_base}/json_employees")
emp_df.write.mode("overwrite").csv(f"{output_base}/csv_employees", header=True)
emp_df.write.mode("overwrite").parquet(f"{output_base}/parquet_employees")

print("Write completed to:", output_base)

# 19. Common interview classification: Transformation vs Action

## Transformations
- `select`
- `selectExpr`
- `withColumn`
- `withColumnRenamed`
- `drop`
- `filter`
- `where`
- `distinct`
- `dropDuplicates`
- `limit`
- `sample`
- `sampleBy`
- `sort`
- `orderBy`
- `sortWithinPartitions`
- `groupBy().agg(...)`
- `rollup`
- `cube`
- `pivot`
- `join`
- `crossJoin`
- `union`
- `unionByName`
- `intersect`
- `intersectAll`
- `exceptAll`
- `subtract`
- `na.drop`
- `na.fill`
- `na.replace`
- `repartition`
- `coalesce`
- `repartitionByRange`
- `cache`
- `persist`
- `hint`
- `transform`
- `replace`
- `createOrReplaceTempView` (view registration operation, usually discussed separately)
- `createGlobalTempView` (view registration operation, usually discussed separately)

## Actions
- `show`
- `count`
- `collect`
- `first`
- `head`
- `take`
- `toLocalIterator`
- `foreach`
- `foreachPartition`
- `describe`
- `summary`
- `toPandas`
- `write.csv`
- `write.json`
- `write.parquet`
- `write.orc`
- `write.jdbc`
- `save`
- `saveAsTable`

## Common helper expressions, not standalone DataFrame transformations
- `col`
- `lit`
- `expr`
- `when`
- `otherwise`
- `alias`
- `cast`
- `asc`
- `desc`
- `row_number`
- `rank`
- `dense_rank`
- `lead`
- `lag`
- `sum`
- `avg`
- `min`
- `max`
- `count`
- `explode`
- `posexplode`
- `inline`
- `array`
- `struct`
- `create_map`
- `concat`
- `regexp_extract`
- `to_date`
- `date_format`
and many more from `pyspark.sql.functions`.

# 20. Real project patterns

## Pattern 1: Clean source columns
`select`, `withColumn`, `cast`, `drop`

## Pattern 2: Filter only valid records
`filter`, `where`, `dropDuplicates`, `na.drop`

## Pattern 3: Enrich with reference data
`join`

## Pattern 4: Aggregate for reporting
`groupBy`, `agg`, `pivot`

## Pattern 5: Deduplicate latest record
window with `row_number()`

## Pattern 6: Handle nested JSON/API data
`explode`, `struct`, `array`, field extraction

## Pattern 7: Improve performance
`repartition`, `coalesce`, `cache`, `broadcast hint`

# 21. Practice exercises

1. Find top 2 highest-paid employees in each department.  
2. Count employees by city and department.  
3. Replace null grades with `"Not Assigned"`.  
4. Join employees with departments and show region.  
5. Create a `salary_band` column with 3 categories.  
6. Deduplicate rows from `null_dup_df`.  
7. Pivot average salary by city and department.  
8. Explode `skills` from `nested_df`.  
9. Find employees who did not receive a bonus using left anti join.  
10. Repartition by `dept` and compare partition count.

# 22. Final summary

This notebook includes the major Spark DataFrame transformations and actions you are expected to know for:
- Spark fundamentals
- PySpark interviews
- AWS Glue
- EMR
- Databricks
- real ETL pipelines

